In [ ]:
import os
import time
import numpy as np
from pathlib import Path
import pickle
from loguru import logger
from numpy.typing import NDArray

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patheffects as pathefx
from numpy.random import default_rng
import scipy.io
from scipy.interpolate import make_interp_spline

from IPython.display import Audio, display
from raves import raves, run_ART, run_MoDART
from raves.src.compute_MoDART import plot_T60
from raves.src.utils.raves_io import visualize_mesh

from slope2noise.utils import db, schroeder_backward_int, octave_filtering

In [ ]:
# depth, length, height
ROOM1_DIMS = [4.5, 18.0, 2.8]
ROOM2_DIMS = [4.6, 6.6, 2.8]

def dataset_to_sim_pos(true_pos: NDArray) -> NDArray[np.float64]:
    """Map ERTD dataset mic position to simulation coordinates."""
    corr_pos = true_pos.copy()
    corr_pos[..., 0] = ROOM1_DIMS[0] + ROOM2_DIMS[0] - corr_pos[..., 0]
    return corr_pos

I have created a custom mesh object for the two coupled rooms in the Extended Room Transition dataset (with the help of Codex). I am going to load this environment and run ART on this and then compare the EDCs of the true and simulated RIRs.

In [ ]:
patch_area = 2.0
environment_name = f'ERTD_1_patch_per_wall'
environment_folder = os.path.join('..', 'environment', environment_name)

In [ ]:
image = visualize_mesh(environment_folder, interactive_window=False)

if image is not None:
    plt.figure()
    plt.imshow(image)
    plt.axis('off')
    plt.show()

### Preprocess to get environment surface to surface matrices and delays

In [ ]:
# the area threshold ensures that surface patches are merged until all areas are above this threshold in m^2
raves(environment_folder, overwrite=True, skip_MoDART=True ,skip_T60_plots=True)

### Select GT RIRs, source and listener positions from the dataset

In [ ]:
data_folder = Path('../../../Data/ERTD_dataset/ertd.pkl').resolve()
with open(data_folder, "rb") as f:
    data_dict = pickle.load(f)

recPos = np.asarray(data_dict['recPos'])
rec_idx_pos = np.argwhere(recPos[1, :]==9.7).squeeze()
selected_rec_pos = recPos[:, rec_idx_pos].T
rirs = np.asarray(data_dict["rirs"], dtype=np.float64)
rir_sample_rate = 44100

source_position = dataset_to_sim_pos(np.asarray(data_dict['srcPos']))[np.newaxis, :]
listener_positions = dataset_to_sim_pos(selected_rec_pos[::3, :])
gt_rirs = rirs[rec_idx_pos[::3], :]

# this RIR has a clear direct path
direct_path_rcx_idx = 2
direct_path_dist = np.linalg.norm(source_position.squeeze() -
                                  listener_positions[direct_path_rcx_idx, :])
gt_rirs_direct_gain = np.max(np.abs(gt_rirs[direct_path_rcx_idx, :]))
# correct 1/r scaling in meters
gt_rirs_exp_direct_gain = 1.0 / direct_path_dist
# print for debugging
logger.warning(f"FDTD direct path gain for receiver at position {np.round(listener_positions[direct_path_rcx_idx, :], 3)} "
    f"and source at position {np.round(source_position.squeeze(), 3)} " \
    f"is {gt_rirs_direct_gain}. Correct gain is {gt_rirs_exp_direct_gain:.5f}.")
# normalise for correct direct path
gt_rirs = gt_rirs * (gt_rirs_exp_direct_gain / gt_rirs_direct_gain)
rir_time_axis = np.arange(0, gt_rirs.shape[-1] / rir_sample_rate, 1.0 / rir_sample_rate)


### Run TD-ART

In [ ]:
# Duration of the impulse responses to be generated, in seconds.
response_duration = gt_rirs.shape[-1] / rir_sample_rate
# Sample rate used for the echograms. Mostly relevant to avoid rounding errors in the propagation delays.
echogram_sample_rate = 1e4

start_time = time.time()
ART_echograms, frequencies = run_ART(environment_folder, source_position, listener_positions,
                                     echogram_sample_rate=echogram_sample_rate,
                                     echogram_duration=response_duration)
ART_runtime = time.time() - start_time

### Plot EDCs against ground truth

In [ ]:
displayed_duration = response_duration / 2

# Prepare a time axis for the plots.
echogram_time_axis = np.arange(0, response_duration, 1 / echogram_sample_rate)
num_bands = len(frequencies)
num_listeners = len(listener_positions)
# Number of (echogram) samples actually shown displayed the plots:
displayed_len_echo = min(len(echogram_time_axis), int(displayed_duration * echogram_sample_rate))

displayed_len_rir = min(len(rir_time_axis), int(displayed_duration * rir_sample_rate))

In [ ]:
ART_EDCs = np.cumsum(ART_echograms[..., ::-1], axis=-1).squeeze()[..., ::-1]
ART_EDCs_dB = db(ART_EDCs, is_squared=True)
GT_EDCs = schroeder_backward_int(gt_rirs, time_axis=-1)
GT_EDCs_dB = db(GT_EDCs, is_squared=True)

# Consider the extent of the dB range to be plotted.
max_extent = max(np.max(ART_EDCs_dB[:, 0]),
                 np.max(GT_EDCs_dB[:,  0]))
min_extent = min(np.min(ART_EDCs_dB[:, displayed_len_echo]),
                 np.min(GT_EDCs_dB[:, displayed_len_rir]))

fig, ax = plt.subplots(4, 4,
                       figsize=(16, 12),   # adjust if needed
                       sharex=True,
                       sharey=True)

ax = ax.flatten()  # make indexing simple

for l in range(num_listeners):
    ax[l].plot(echogram_time_axis, ART_EDCs_dB[l],
               label='TD-ART', marker='o', fillstyle='none',
               markevery=int(3e-2 * echogram_sample_rate))

    ax[l].plot(rir_time_axis, GT_EDCs_dB[l],
               label='Ground truth', marker='x',
               markevery=int(3e-2 * echogram_sample_rate))

    ax[l].set_xlim(0, displayed_duration)
    ax[l].set_ylim(min_extent, max_extent + 3)
    ax[l].set_title(f' L{l+1}')

    # Only left column gets y-label
    if l % 4 == 0:
        ax[l].set_ylabel('EDC [dB]')

    # Only bottom row gets x-label
    if l >= 12:
        ax[l].set_xlabel('Time [s]')

# Put legend only once (top-right subplot for example)
ax[0].legend()

plt.suptitle(f'Runtime: TD-ART {ART_runtime:.3f}s.')
plt.tight_layout()
fig_path = Path('../../../Figures/CV-SDN/ART')
fig_path.mkdir(parents=True, exist_ok=True)
plt.savefig(f'{fig_path.resolve()}/{environment_name}_compare_EDC.png')
plt.show()

### Convert from energy to amplitude for a fairer EDC comparison

In [ ]:
# Take note of the echogram energy, to compare it after upsampling.
old_energy = np.sum(ART_echograms, axis=-1)

# We use a linear interpolation, because any other upsampling algorithm risks introducing negative values.
linear_spline = make_interp_spline(echogram_time_axis, ART_echograms.squeeze(axis=-2), k=1, axis=-1)
upsampled_echograms = linear_spline(rir_time_axis)

# Normalize w.r.t. the new sample rate, to preserve the energy-per-second definition of echogram values.
upsampled_echograms *= echogram_sample_rate / rir_sample_rate

# Compare the new energy to the old one.
new_energy = np.sum(upsampled_echograms, axis=-1)
# The ratio (averaged over all frequency bands) should be close to 1 for all sources and listeners.
print(np.mean(old_energy / new_energy, axis=-1))

# Random number generator.
rng = default_rng()

# White noise
noise_signal = np.random.randn(len(rir_time_axis))

# Normalize energy per second.
noise_signal *= np.sqrt(response_duration / np.sum(noise_signal**2))

# Translate the energy envelopes to amplitude envelopes.
envelopes = np.sqrt(upsampled_echograms)

# The envelope array has shape (S, L, T), the noise signals have shape (T):
#   we need to add two "leading" dimensions, which is done using [None, None].
modulated_noise_signal = envelopes * noise_signal[None, None]

In [ ]:
ART_synth_EDCs = schroeder_backward_int(modulated_noise_signal, time_axis=-1).squeeze()
ART_synth_EDCs_dB = db(ART_synth_EDCs, is_squared=True)

# Consider the extent of the dB range to be plotted.
max_extent = max(np.max(ART_synth_EDCs_dB[:, 0]),
                 np.max(GT_EDCs_dB[:,  0]))
min_extent = min(np.min(ART_synth_EDCs_dB[:, displayed_len_rir]),
                 np.min(GT_EDCs_dB[:, displayed_len_rir]))

fig, ax = plt.subplots(4, 4,
                       figsize=(16, 12),   # adjust if needed
                       sharex=True,
                       sharey=True)

ax = ax.flatten()  # make indexing simple

for l in range(num_listeners):
    ax[l].plot(rir_time_axis, ART_synth_EDCs_dB[l],
               label='TD-ART')

    ax[l].plot(rir_time_axis, GT_EDCs_dB[l],
               label='Ground truth')

    ax[l].set_xlim(0, displayed_duration)
    ax[l].set_ylim(min_extent, max_extent + 3)
    ax[l].set_title(f' L{l+1}')

    # Only left column gets y-label
    if l % 4 == 0:
        ax[l].set_ylabel('EDC [dB]')

    # Only bottom row gets x-label
    if l >= 12:
        ax[l].set_xlabel('Time [s]')

# Put legend only once (top-right subplot for example)
ax[0].legend()

plt.suptitle(f'Runtime: TD-ART {ART_runtime:.3f}s.')
plt.tight_layout()
fig_path = Path('../../../Figures/CV-SDN/ART')
fig_path.mkdir(parents=True, exist_ok=True)
plt.savefig(f'{fig_path.resolve()}/{environment_name}_compare_synth_RIR_EDC.png')
plt.show()

### Plot ART matrix for each surface patch

Since we have 16 patches, and there is no connection from the patch to itself, we'd expect the size of this matrix to be (16*15 x 16*15). However, many patches are not visible from each other. The patching matrix is 16 x 16 and determines what patches are visible from each other. 

In [ ]:
# Read the .mtx file
art_matrix_diffuse = scipy.io.mmread(f'{environment_folder}/ART_kernel_diffuse.mtx')
art_matrix_specular = scipy.io.mmread(f'{environment_folder}/ART_kernel_specular.mtx')
art_reflect_matrix = scipy.io.mmread(f'{environment_folder}/ART_kernel_band_1.mtx')
art_patching_matrix = scipy.io.mmread(f'{environment_folder}/path_indexing.mtx')

# Convert to dense format if needed for visualization (optional)
art_dense_matrix = art_reflect_matrix.todense()
art_patching_matrix_dense = art_patching_matrix.todense()
num_non_zero_elems = np.count_nonzero(art_patching_matrix_dense)
assert art_dense_matrix.shape[0] == num_non_zero_elems

# Create a Spy plot to visualize sparsity
plt.figure(figsize=(5, 5))
plt.spy(art_patching_matrix, markersize=0.5)
plt.title('Patching Matrix Sparsity Pattern')
plt.xlabel('Patch #')
plt.ylabel('Patch #')
plt.savefig(f'{fig_path.resolve()}/{environment_name}_patching_matrix.png')
plt.show()

# Create a Heatmap for density
plt.figure(figsize=(10, 8))
plt.imshow(art_dense_matrix, cmap='viridis', interpolation='nearest')
plt.colorbar()
plt.title('ART matrix heatmap')
plt.xlabel('Out. chans (idx of ij)')
plt.ylabel('Inc. chans (idx of hi)')
plt.savefig(f'{fig_path.resolve()}/{environment_name}_ART_matrix.png')
plt.show()
